In [1]:
#!/usr/bin/env python3
#"""Nonsingular terminal sliding-mode control of a thrust AMB.

#The bias-linearized plant requested by the user is

#                    k_i K_amp
#    G(s) = --------------------------------,
#           (m s**2 - k_x) (tau s + 1)

#with the state realization

#    x_dot = v
#    m v_dot = k_x*x + k_i*i_c + F_d
#    tau i_c_dot = -i_c + K_amp*V_sp.

#``k_x`` is stored as the positive magnitude of the destabilizing stiffness in
#``m*x_ddot-k_x*x``.  If a source reports a signed negative coefficient, enter
#its magnitude.  Position ``x`` is displacement from the nominal air gap.

#The controller uses a nonsingular terminal sliding surface and a cascaded
#current loop.  A smooth saturation boundary layer suppresses coil-current
#chattering.  In the ideal noise-free/sign-function case the reaching law gives
#finite-time convergence; with the practical boundary layer it gives finite-time
#entry into a configurable invariant error tube.

#Dependencies: numpy, scipy, matplotlib
#Run in a terminal: python Cho2007_Terminal_Sliding_Mode_Thrust_AMB.py
#Run in Jupyter:   %run Cho2007_Terminal_Sliding_Mode_Thrust_AMB.py
#"""
from __future__ import annotations

import argparse
import json
from collections import deque
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable, Deque, Dict, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal


@dataclass
class AMBPlantParameters:
#    """Replace these placeholder values with identified experimental values."""

    rotor_mass: float = 0.42
    negative_stiffness_magnitude: float = 120.0
    current_force_gain: float = 3.8
    amplifier_gain: float = 0.45
    amplifier_time_constant: float = 0.006
    nominal_air_gap: float = 0.014
    bias_current: float = 2.20

    control_current_min: float = -1.80
    control_current_max: float = 1.80
    voltage_setpoint_limit: float = 10.0
    dc_bus_voltage: float = 48.0
    pwm_duty_limit: float = 0.95
    amplifier_efficiency: float = 0.97

    minimum_air_gap: float = 0.010
    maximum_air_gap: float = 0.018
    initial_displacement: float = 0.25e-3
    initial_velocity: float = 0.0

    def effective_voltage_limit(self) -> float:
        return min(self.voltage_setpoint_limit,
                   self.dc_bus_voltage * self.pwm_duty_limit)

    def analytical_tf(self) -> Tuple[np.ndarray, np.ndarray]:
        m = self.rotor_mass
        kx = self.negative_stiffness_magnitude
        tau = self.amplifier_time_constant
        return (np.array([self.current_force_gain * self.amplifier_gain]),
                np.array([m * tau, m, -kx * tau, -kx]))


@dataclass
class SensorParameters:
    sensitivity: float = 800.0
    bias_voltage: float = 0.0008
    noise_variance: float = 4.0e-7
    lowpass_cutoff_hz: float = 100.0


@dataclass
class TerminalControllerParameters:
#    """Digital nonsingular terminal SMC and inner current-loop settings."""
    sample_time: float = 0.0005
    # Surface s=e+beta*sig(e_dot)^gamma; 1<gamma<2 is nonsingular TSMC.
    surface_beta: float = 0.050
    surface_exponent: float = 1.5
    reaching_linear_gain: float = 30.0
    reaching_switch_gain: float = 0.003
    boundary_layer: float = 0.030e-3
    velocity_filter_cutoff_hz: float = 40.0
    reference_filter_bandwidth_rad_s: float = 35.0
    current_loop_bandwidth_rad_s: float = 120.0
    derivative_regularization: float = 1.0e-3
    maximum_current_command_slew_rate: float = 100.0  # A/s
    computational_delay_samples: int = 0
    maximum_voltage_slew_rate: float = 2500.0  # V/s


@dataclass
class SimulationParameters:
    duration: float = 3.0
    integration_substeps: int = 4
    disturbance_onset: float = 0.5
    disturbance_settling_band: float = 0.10e-3
    disturbance_settling_hold_time: float = 0.15


class ThrustAMBPlant:
#    """Saturated RK4 realization of the exact third-order transfer model."""

    def __init__(self, p: AMBPlantParameters):
        self.p = p
        self.state = np.array([p.initial_displacement, p.initial_velocity, 0.0])
        self.voltage_applied = 0.0
        self.duty_ratio = 0.0

    def limited_voltage(self, demand: float) -> float:
        command = float(np.clip(demand, -self.p.effective_voltage_limit(),
                                self.p.effective_voltage_limit()))
        self.duty_ratio = command / self.p.dc_bus_voltage
        return self.p.amplifier_efficiency * command

    def derivative(self, state: np.ndarray, voltage_demand: float,
                   load_force: float, apply_limits: bool = True) -> np.ndarray:
        p = self.p
        x, velocity, incremental_current = state
        voltage = self.limited_voltage(voltage_demand) if apply_limits else voltage_demand
        current_target = p.amplifier_gain * voltage
        if apply_limits:
            current_target = float(np.clip(current_target, p.control_current_min,
                                           p.control_current_max))
        current_dot = (current_target - incremental_current) / p.amplifier_time_constant
        if apply_limits and ((incremental_current <= p.control_current_min and current_dot < 0)
                             or (incremental_current >= p.control_current_max and current_dot > 0)):
            current_dot = 0.0
        acceleration = (p.negative_stiffness_magnitude * x
                        + p.current_force_gain * incremental_current
                        + load_force) / p.rotor_mass
        return np.array([velocity, acceleration, current_dot])

    def step(self, voltage_demand: float, load_force: float, dt: float,
             substeps: int = 4) -> None:
        h = dt / substeps
        for _ in range(substeps):
            s = self.state
            k1 = self.derivative(s, voltage_demand, load_force)
            k2 = self.derivative(s + 0.5*h*k1, voltage_demand, load_force)
            k3 = self.derivative(s + 0.5*h*k2, voltage_demand, load_force)
            k4 = self.derivative(s + h*k3, voltage_demand, load_force)
            self.state = s + h*(k1 + 2*k2 + 2*k3 + k4)/6.0
        self.state[2] = np.clip(self.state[2], self.p.control_current_min,
                                self.p.control_current_max)
        self.voltage_applied = self.limited_voltage(voltage_demand)


class SensorFilter:
#    """Position sensitivity, offset, Gaussian noise, and 2nd-order Butterworth."""

    def __init__(self, p: SensorParameters, sample_time: float,
                 initial_displacement: float, seed: int = 21):
        if p.lowpass_cutoff_hz >= 0.5/sample_time:
            raise ValueError("Sensor cutoff must be below the Nyquist frequency.")
        self.p = p
        self.rng = np.random.default_rng(seed)
        self.sos = signal.butter(2, p.lowpass_cutoff_hz, fs=1/sample_time,
                                 output="sos")
        initial_voltage = p.sensitivity*initial_displacement + p.bias_voltage
        self.zi = signal.sosfilt_zi(self.sos) * initial_voltage

    def measure(self, displacement: float) -> float:
        voltage = (self.p.sensitivity*displacement + self.p.bias_voltage
                   + self.rng.normal(0.0, np.sqrt(self.p.noise_variance)))
        filtered, self.zi = signal.sosfilt(self.sos, [voltage], zi=self.zi)
        # Dividing by sensitivity leaves residual sensor bias visible to TSMC.
        return float(filtered[0] / self.p.sensitivity)


class TerminalController:
#    """Nonsingular terminal SMC with smooth switching and current control.

#    Let e=r-x and define ``sig(v)^a = sign(v)|v|^a``.  The surface is

#        s = e + beta*sig(e_dot)^gamma,       1 < gamma < 2.

#    This is the nonsingular form of terminal SMC: solving its derivative for
#    desired error acceleration does not divide by a fractional power of e.
#    The reaching law

#        s_dot = -k*s - eta*sat(s/phi)

#    gives V=|s| the inequality V_dot <= -eta outside the boundary layer;
#    hence the ideal reaching-time bound is |s(0)|/eta.  ``sat`` replaces
#    ``sign`` inside |s|<=phi to suppress chattering, so the implemented noisy
#    system guarantees practical finite-time entry into a tube, not exact zero.

#    The outer law calculates desired acceleration, cancels known negative
#    stiffness, and commands coil current.  A model-inverting inner loop uses
#    tau*i_dot=-i+Kamp*efficiency*Vsp and adds first-order current feedback.
#    """

    def __init__(self, c: TerminalControllerParameters, plant: AMBPlantParameters):
        if not 1.0 < c.surface_exponent < 2.0:
            raise ValueError("surface_exponent must satisfy 1 < gamma < 2")
        if c.surface_beta <= 0 or c.boundary_layer <= 0:
            raise ValueError("surface_beta and boundary_layer must be positive")
        self.c, self.p = c, plant
        self.voltage_limit = plant.effective_voltage_limit()
        self.reference_state = np.zeros(2)
        self.velocity_estimate = 0.0
        self.previous_measurement: Optional[float] = None
        self.previous_current_command = 0.0
        self.previous_voltage = 0.0

    @staticmethod
    def _rk4(state: np.ndarray, dt: float,
             rhs: Callable[[np.ndarray], np.ndarray]) -> np.ndarray:
        k1 = rhs(state)
        k2 = rhs(state + 0.5*dt*k1)
        k3 = rhs(state + 0.5*dt*k2)
        k4 = rhs(state + dt*k3)
        return state + dt*(k1 + 2*k2 + 2*k3 + k4)/6.0

    def update(self, reference: float, measurement: float,
               measured_incremental_current: float) -> Tuple[float, Dict[str, float]]:
        dt = self.c.sample_time
        wr = self.c.reference_filter_bandwidth_rad_s

        # Critically damped reference filter supplies r, r_dot, and r_ddot.
        def reference_rhs(q: np.ndarray) -> np.ndarray:
            return np.array([q[1], -2*wr*q[1]-wr**2*(q[0]-reference)])
        self.reference_state = self._rk4(self.reference_state, dt, reference_rhs)
        reference_acceleration = (-2*wr*self.reference_state[1]
                                  - wr**2*(self.reference_state[0]-reference))

        if self.previous_measurement is None:
            raw_velocity = 0.0
        else:
            raw_velocity = (measurement-self.previous_measurement)/dt
        self.previous_measurement = measurement
        # First-order derivative LPF prevents sensor-noise amplification.
        alpha = np.exp(-2*np.pi*self.c.velocity_filter_cutoff_hz*dt)
        self.velocity_estimate = alpha*self.velocity_estimate+(1-alpha)*raw_velocity

        error = self.reference_state[0]-measurement
        error_rate = self.reference_state[1]-self.velocity_estimate
        gamma, beta = self.c.surface_exponent, self.c.surface_beta
        signed_power = np.sign(error_rate)*abs(error_rate)**gamma
        surface = error+beta*signed_power
        sat_surface = float(np.clip(surface/self.c.boundary_layer, -1.0, 1.0))

        # Regularization removes numerical ill-conditioning near e_dot=0.  It
        # is deliberately much smaller than normal controlled velocities.
        denominator = beta*gamma*(abs(error_rate)+self.c.derivative_regularization)**(gamma-1)
        desired_error_acceleration = (
            -error_rate-self.c.reaching_linear_gain*surface
            -self.c.reaching_switch_gain*sat_surface) / denominator

        # From e_ddot=r_ddot-(kx*x+ki*i+Fd)/m, omit unknown Fd and let the
        # robust reaching term dominate its bounded contribution.
        desired_current_unlimited = (
            self.p.rotor_mass*(reference_acceleration-desired_error_acceleration)
            -self.p.negative_stiffness_magnitude*measurement
        )/self.p.current_force_gain
        desired_current_unlimited = float(np.clip(
            desired_current_unlimited, self.p.control_current_min,
            self.p.control_current_max))
        max_di = self.c.maximum_current_command_slew_rate*dt
        desired_current = float(np.clip(desired_current_unlimited,
                                        self.previous_current_command-max_di,
                                        self.previous_current_command+max_di))
        desired_current_rate = (desired_current-self.previous_current_command)/dt
        self.previous_current_command = desired_current

        # Exact nominal amplifier inversion plus stabilizing current feedback.
        effective_gain = self.p.amplifier_gain*self.p.amplifier_efficiency
        unlimited = (measured_incremental_current + self.p.amplifier_time_constant*(
            desired_current_rate+self.c.current_loop_bandwidth_rad_s*(
                desired_current-measured_incremental_current))) / effective_gain
        max_change = self.c.maximum_voltage_slew_rate * dt
        slew_limited = np.clip(unlimited, self.previous_voltage-max_change,
                               self.previous_voltage+max_change)
        command = float(np.clip(slew_limited, -self.voltage_limit,
                                self.voltage_limit))
        self.previous_voltage = command
        return command, {"filtered_reference": self.reference_state[0],
                         "filtered_reference_rate": self.reference_state[1],
                         "estimated_velocity": self.velocity_estimate,
                         "error": error, "error_rate": error_rate,
                         "sliding_surface": surface,
                         "desired_current": desired_current,
                         "unlimited_voltage": unlimited}


def default_reference(_: float) -> float:
    return 0.0


def default_disturbance(t: float) -> float:
#    """Step, sinusoidal surge, and finite-duration pulse in newtons."""
    step = 0.55 if t >= 0.5 else 0.0
    sine = 0.10*np.sin(2*np.pi*1.2*(t-1.5)) if t >= 1.5 else 0.0
    pulse = -0.30 if 2.15 <= t < 2.22 else 0.0
    return step + sine + pulse


class SimulationEnvironment:
    def __init__(self, plant_parameters: AMBPlantParameters,
                 sensor_parameters: SensorParameters,
                 controller_parameters: TerminalControllerParameters,
                 simulation_parameters: SimulationParameters,
                 seed: int = 21,
                 reference: Callable[[float], float] = default_reference,
                 disturbance: Callable[[float], float] = default_disturbance):
        self.pp, self.sp, self.cp, self.sim = (plant_parameters, sensor_parameters,
                                               controller_parameters, simulation_parameters)
        self.plant = ThrustAMBPlant(self.pp)
        self.sensor = SensorFilter(self.sp, self.cp.sample_time,
                                   self.pp.initial_displacement, seed)
        self.controller = TerminalController(self.cp, self.pp)
        self.reference, self.disturbance = reference, disturbance

    def run(self) -> Dict[str, np.ndarray]:
        dt = self.cp.sample_time
        time = np.arange(0.0, self.sim.duration + 0.5*dt, dt)
        names = ("position", "velocity", "measurement", "reference", "gap",
                 "incremental_current", "total_current", "voltage_demand",
                 "voltage_applied", "load_force", "error", "error_rate",
                 "sliding_surface", "desired_current", "estimated_velocity",
                 "filtered_reference", "filtered_reference_rate",
                 "unlimited_voltage")
        log = {name: np.zeros(time.size) for name in names}
        delay: Deque[float] = deque([0.0]*(self.cp.computational_delay_samples+1),
                                    maxlen=self.cp.computational_delay_samples+1)
        for k, t in enumerate(time):
            x, velocity, current = self.plant.state
            measured = self.sensor.measure(x)
            reference = self.reference(float(t))
            demand, controller_data = self.controller.update(reference, measured, current)
            delay.append(demand)
            delayed_demand = delay[0]
            load = self.disturbance(float(t))

            log["position"][k], log["velocity"][k] = x, velocity
            log["measurement"][k], log["reference"][k] = measured, reference
            log["gap"][k] = self.pp.nominal_air_gap + x
            log["incremental_current"][k] = current
            log["total_current"][k] = self.pp.bias_current + current
            log["voltage_demand"][k] = demand
            log["voltage_applied"][k] = self.plant.voltage_applied
            log["load_force"][k] = load
            for key in ("error", "error_rate", "sliding_surface", "desired_current",
                        "estimated_velocity", "filtered_reference",
                        "filtered_reference_rate", "unlimited_voltage"):
                log[key][k] = controller_data[key]

            self.plant.step(delayed_demand, load, dt,
                            self.sim.integration_substeps)

        log["time"] = time
        return log

    def metrics(self, d: Dict[str, np.ndarray]) -> Dict[str, Optional[float]]:
        error = d["reference"] - d["position"]
        settling = disturbance_rejection_settling_time(
            d["time"], error, self.sim.disturbance_onset,
            self.sim.disturbance_settling_band,
            self.sim.disturbance_settling_hold_time)
        start = int(np.searchsorted(d["time"], self.sim.disturbance_onset))
        if settling is None or settling <= 0:
            convergence_rate = None
        else:
            stop = int(np.searchsorted(d["time"], self.sim.disturbance_onset+settling))+1
            peak_after_load = float(np.max(np.abs(error[start:stop])))
            convergence_rate = 1e6*peak_after_load/settling
        return {
            "position_RMSE_um": float(1e6*np.sqrt(np.mean(error**2))),
            "peak_gap_deviation_um": float(1e6*np.max(np.abs(error))),
            "maximum_incremental_current_A": float(np.max(np.abs(d["incremental_current"]))),
            "maximum_total_current_A": float(np.max(np.abs(d["total_current"]))),
            "maximum_voltage_demand_V": float(np.max(np.abs(d["voltage_demand"]))),
            "current_chattering_index_mA_RMS_per_sample": float(
                1e3*np.sqrt(np.mean(np.diff(d["incremental_current"])**2))),
            "finite_time_convergence_time_s": settling,
            "finite_time_convergence_rate_um_per_s": convergence_rate,
            "maximum_absolute_sliding_surface_um": float(
                1e6*np.max(np.abs(d["sliding_surface"]))),
            "minimum_gap_mm": float(1e3*np.min(d["gap"])),
            "maximum_gap_mm": float(1e3*np.max(d["gap"])),
            "safety_gap_violation": bool(np.any(d["gap"] < self.pp.minimum_air_gap)
                                         or np.any(d["gap"] > self.pp.maximum_air_gap)),
        }


def disturbance_rejection_settling_time(time: np.ndarray, error: np.ndarray,
                                         onset: float, band: float,
                                         hold_time: float) -> Optional[float]:
    dt = float(time[1]-time[0])
    hold = max(1, int(np.ceil(hold_time/dt)))
    start = int(np.searchsorted(time, onset))
    inside = np.abs(error) <= band
    for k in range(start, len(time)-hold+1):
        if np.all(inside[k:k+hold]):
            return float(time[k]-onset)
    return None


def validate_transfer_function(p: AMBPlantParameters) -> float:
#    """Numerically validate the unsaturated RK4 realization against scipy TF."""
    dt, duration, amplitude = 2e-5, 0.12, 0.05
    time = np.arange(0.0, duration+0.5*dt, dt)
    num, den = p.analytical_tf()
    _, analytical = signal.step(signal.TransferFunction(num, den), T=time)
    analytical *= amplitude
    state = np.zeros(3)
    numerical = np.zeros_like(time)

    def rhs(q: np.ndarray) -> np.ndarray:
        x, v, current = q
        return np.array([v,
            (p.negative_stiffness_magnitude*x+p.current_force_gain*current)/p.rotor_mass,
            (p.amplifier_gain*amplitude-current)/p.amplifier_time_constant])

    for k in range(time.size-1):
        numerical[k] = state[0]
        k1 = rhs(state); k2 = rhs(state+0.5*dt*k1)
        k3 = rhs(state+0.5*dt*k2); k4 = rhs(state+dt*k3)
        state += dt*(k1+2*k2+2*k3+k4)/6
    numerical[-1] = state[0]
    scale = max(np.max(np.abs(analytical)), np.finfo(float).eps)
    return float(100*np.sqrt(np.mean((numerical-analytical)**2))/scale)


def plot_results(d: Dict[str, np.ndarray], p: AMBPlantParameters,
                 c: TerminalControllerParameters, output: Path) -> None:
    t = d["time"]
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
    axes = axes.ravel()
    axes[0].plot(t, 1e3*d["gap"], label="actual air gap", lw=1.5)
    axes[0].plot(t, 1e3*(p.nominal_air_gap+d["reference"]), "--", label="reference")
    axes[0].axhline(1e3*p.minimum_air_gap, color="r", ls=":", label="safety bounds")
    axes[0].axhline(1e3*p.maximum_air_gap, color="r", ls=":")
    axes[0].set_ylabel("Gap [mm]"); axes[0].legend(ncol=3); axes[0].grid(True)

    axes[1].plot(t, d["incremental_current"], label="incremental current")
    axes[1].plot(t, d["desired_current"], "--", alpha=.65,
                 label="slew-limited current command")
    axes[1].axhline(p.control_current_max, color="tab:blue", ls=":")
    axes[1].axhline(p.control_current_min, color="tab:blue", ls=":")
    axes[1].set_ylabel("Current [A]"); axes[1].grid(True)
    voltage_axis = axes[1].twinx()
    voltage_axis.plot(t, d["voltage_demand"], color="tab:orange", alpha=.8,
                      label="voltage demand")
    voltage_axis.axhline(p.effective_voltage_limit(), color="tab:orange", ls=":")
    voltage_axis.axhline(-p.effective_voltage_limit(), color="tab:orange", ls=":")
    voltage_axis.set_ylabel("Voltage [V]")
    lines = axes[1].lines[:2] + voltage_axis.lines[:1]
    axes[1].legend(lines, [line.get_label() for line in lines], loc="upper right")

    axes[2].plot(t, 1e6*d["sliding_surface"], label="sliding surface")
    axes[2].axhspan(-1e6*c.boundary_layer, 1e6*c.boundary_layer,
                    color="tab:green", alpha=.15, label="boundary layer")
    axes[2].set_ylabel(r"Surface $s$ [$\mu$m]"); axes[2].set_xlabel("Time [s]")
    axes[2].legend(loc="upper right"); axes[2].grid(True)
    load_axis = axes[2].twinx()
    load_axis.plot(t, d["load_force"], color="tab:red", alpha=.45,
                   label="axial load")
    load_axis.set_ylabel("Load [N]")

    phase = axes[3].scatter(1e6*d["error"], 1e3*d["error_rate"], c=t,
                            s=3, cmap="viridis", rasterized=True)
    axes[3].scatter([0], [0], marker="x", color="red", s=50, label="origin")
    axes[3].set_xlabel(r"Error $e$ [$\mu$m]")
    axes[3].set_ylabel(r"Error rate $\dot e$ [mm/s]")
    axes[3].set_title("Terminal phase portrait"); axes[3].grid(True)
    axes[3].legend(); fig.colorbar(phase, ax=axes[3], label="Time [s]")
    fig.suptitle("Cho (2007) thrust AMB — terminal sliding-mode validation",
                 fontsize=14)
    fig.savefig(output, dpi=180)
    plt.close(fig)


def parse_arguments() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Cho-model thrust AMB terminal SMC simulation")
    parser.add_argument("--output", type=Path, default=Path("cho2007_tsmc_results"))
    parser.add_argument("--seed", type=int, default=21)
    # parse_known_args is deliberate: Jupyter injects a '-f kernel.json' pair.
    args, _ = parser.parse_known_args()
    return args


def main() -> None:
    args = parse_arguments()
    args.output.mkdir(parents=True, exist_ok=True)
    plant = AMBPlantParameters()
    sensor = SensorParameters()
    controller = TerminalControllerParameters()
    simulation = SimulationParameters()

    tf_error = validate_transfer_function(plant)
    environment = SimulationEnvironment(plant, sensor, controller, simulation,
                                        seed=args.seed)
    data = environment.run()
    metrics = environment.metrics(data)
    metrics["transfer_function_validation_normalized_RMSE_percent"] = tf_error

    np.savez_compressed(args.output/"simulation_data.npz", **data)
    plot_results(data, plant, controller, args.output/"terminal_smc_validation.png")
    with (args.output/"metrics.json").open("w", encoding="utf-8") as handle:
        json.dump(metrics, handle, indent=2)
    parameters = {"plant": asdict(plant), "sensor": asdict(sensor),
                  "controller": asdict(controller), "simulation": asdict(simulation)}
    with (args.output/"parameters.json").open("w", encoding="utf-8") as handle:
        json.dump(parameters, handle, indent=2)

    print("\nCho (2007) thrust AMB — terminal SMC validation")
    print("-"*51)
    for key, value in metrics.items():
        print(f"{key:52s}: {value}")
    print(f"\nResults written to: {args.output.resolve()}")


if __name__ == "__main__":
    main()


Cho (2007) thrust AMB — terminal SMC validation
---------------------------------------------------
position_RMSE_um                                    : 67.06279606474347
peak_gap_deviation_um                               : 250.0
maximum_incremental_current_A                       : 0.41439237414298635
maximum_total_current_A                             : 2.4292521395314135
maximum_voltage_demand_V                            : 2.258853690439798
current_chattering_index_mA_RMS_per_sample          : 15.584910355325773
finite_time_convergence_time_s                      : 0.022499999999999964
finite_time_convergence_rate_um_per_s               : 5473.326083091752
maximum_absolute_sliding_surface_um                 : 251.0948987763252
minimum_gap_mm                                      : 13.922244047903584
maximum_gap_mm                                      : 14.25
safety_gap_violation                                : False
transfer_function_validation_normalized_RMSE_percent: 1.0131093